In [ ]:
import pandas as pd
import numpy as np
import os
import h5py
from scipy.io import mmread
from scipy.sparse import csr_matrix
from glob import glob
import shutil
import gzip
import scipy

# Define data path

In [ ]:
dataset = "cos_hsa_sarcoma_c"
datadir = f"PATH/TO/{dataset}/"
outdir = f"{datadir}/h5"

# Export the default CosMx pseudo-spot matrix to H5


In [ ]:
os.makedirs(outdir + "/spatial", exist_ok = True)
# load cnts
exp_data = pd.read_csv(
        f"{datadir}/cnts.tsv",
        sep='\t',
        header=0,
        index_col=0
)
exp_data.index.name = None
# load locs
locs = pd.read_csv(
    f"{datadir}/locs-raw.tsv",
    sep='\t',
    header=0,
    index_col=0
)
locs['in_tissue'] = 1
locs['array_row'] = locs['y']
locs['array_col'] = locs['x']
x = list(locs['x'].values)
y = list(locs['y'].values)
x_keys_sort = sorted(set(x))
y_keys_sort = sorted(set(y))
x_rank_dict = {value: rank for rank, value in enumerate(x_keys_sort)}
y_rank_dict = {value: rank for rank, value in enumerate(y_keys_sort)}
locs = locs.replace({"array_row": y_rank_dict, "array_col": x_rank_dict})
locs["array_row"] = locs["array_row"].astype(int)
locs["array_col"] = locs["array_col"].astype(int)
locs.rename(columns={'x':'pxl_col_in_fullres','y':'pxl_row_in_fullres'},inplace = True)
locs = locs[['in_tissue','array_row','array_col','pxl_row_in_fullres','pxl_col_in_fullres']]
locs[['pxl_col_in_fullres', 'pxl_row_in_fullres']] = locs[['pxl_col_in_fullres', 'pxl_row_in_fullres']].round().astype(int)
locs.index.name = None

locs.to_csv(
    f'{outdir}/spatial/tissue_positions_list.csv', 
    sep=',', 
    index=True,
    header=False
)
locs.to_csv(
    f"{outdir}/spatial/tissue_positions_list_with_header.csv", 
    sep=',', 
    index=True,
    header=True
)

# generate files
features = pd.DataFrame([s.translate(str.maketrans({'/': '_'})) for s in exp_data.columns.values])
features.columns = ['name']
features['id'] = [f"GENE_{i}" for i in range(len(exp_data.columns))]
features['feature_type'] =  "Gene Expression"
features['genome'] = "GRCh38"
features = features[["id", "name", "feature_type","genome"]]
features_columns = features.columns.values
## save matrix.mtx.gz
reAnno_count_sparse_mtx = scipy.sparse.csr_matrix(exp_data.T.values)
with open(os.path.join(outdir, 'matrix.mtx'), "wb") as matrix_file:
    scipy.io.mmwrite(matrix_file, reAnno_count_sparse_mtx)
with open(os.path.join(outdir,'matrix.mtx'),'rb') as mtx_in:
    with gzip.open(os.path.join(outdir,'matrix.mtx') + '.gz','wb') as mtx_gz:
        shutil.copyfileobj(mtx_in, mtx_gz)
os.remove(os.path.join(outdir,'matrix.mtx'))
## save barcodes.tsv.gz
barcodesFile = pd.DataFrame(exp_data.index)
barcodesFile.to_csv(os.path.join(outdir,"barcodes.tsv.gz"),sep='\t',header =False,index=False)
##save features.tsv.gz
features.to_csv(os.path.join(outdir,"features.tsv.gz"),sep='\t',header =False,index=False)

# generate h5
expression_data = mmread(f"{outdir}/matrix.mtx.gz").tocsc()
features = pd.read_csv(f"{outdir}/features.tsv.gz", header=None, sep="\t")
features.columns = features_columns
barcodes = pd.read_csv(f"{outdir}/barcodes.tsv.gz", header=None)
barcodes.columns = ["barcode"]
barcode_list = barcodes["barcode"].values

gene_id_list = features["id"].values
gene_name_list = features["name"].values
gene_type_list = features["feature_type"].values
gene_genome_list = features["genome"]

barcode_list = [str(x) for x in barcode_list]
gene_id_list   = [str(x) for x in gene_id_list]
gene_name_list = [str(x) for x in gene_name_list]
gene_type_list = [str(x) for x in gene_type_list]
gene_genome_list = [str(x) for x in gene_genome_list]

str_dtype = h5py.string_dtype(encoding='utf-8')

# generate H5 file
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", 'w') as f:
    f.attrs['library_ids'] = np.array([dataset], dtype='S')

    group = f.create_group('matrix')

    group.create_dataset('data', data=expression_data.data)
    group.create_dataset('indices', data=expression_data.indices)
    group.create_dataset('indptr', data=expression_data.indptr)
    group.create_dataset('shape', data=expression_data.shape)

    group.create_dataset(
        'barcodes',
        data=np.array(barcode_list, dtype=object),
        dtype=str_dtype
    )

    feature_group = group.create_group('features')

    feature_group.create_dataset(
        'id',
        data=np.array(gene_id_list, dtype=object),
        dtype=str_dtype
    )
    feature_group.create_dataset(
        'name',
        data=np.array(gene_name_list, dtype=object),
        dtype=str_dtype
    )
    feature_group.create_dataset(
        'feature_type',
        data=np.array(gene_type_list, dtype=object),
        dtype=str_dtype
    )
    feature_group.create_dataset(
        'genome',
        data=np.array(gene_genome_list, dtype=object),
        dtype=str_dtype
    )

    feature_group.create_dataset(
        '_all_tag_keys',
        data=np.array(['genome'], dtype=object),
        dtype=str_dtype
    )

    target_group = feature_group.create_group('target_sets')

    target_group.create_dataset(
        'AddOn Panel',
        data=np.arange(len(gene_id_list), dtype=np.int64)
    )


# Inspect the exported H5


In [ ]:
import h5py
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5","r") as f:
    print(f['matrix'].keys())

In [ ]:
with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", "r") as handle:
    print("Gene x spot shape:", tuple(handle["matrix/shape"][:]))